# V24 OCR Recognizer Training — Latin Digits & Punctuation Patch (Baseline: V23)


## Phase 1: Environment Setup


In [ ]:
# Install required libraries
!pip install paddlepaddle-gpu paddleocr opencv-python albumentations pyclipper shapely Pillow pyyaml fonttools rapidfuzz "numpy<2.0.0"

# Verify GPU
import paddle
print("Paddle Version:", paddle.__version__)
print("Paddle compiled with CUDA:", paddle.is_compiled_with_cuda())
print("Available GPU Count:", paddle.device.cuda.device_count())
if paddle.device.cuda.device_count() > 0:
    paddle.utils.run_check()


## Phase 2: Clone PaddleOCR & Directory Structure


In [ ]:
# Clone repo
%cd /kaggle/working/
!git clone -b release/2.7 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -r requirements.txt "numpy<2.0.0"
%cd /kaggle/working/

# Create folders
!mkdir -p /kaggle/working/paddleocr_cham_finetune/data/fonts
!mkdir -p /kaggle/working/paddleocr_cham_finetune/data/corpus
!mkdir -p /kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27
!mkdir -p /kaggle/working/paddleocr_cham_finetune/scripts
!mkdir -p /kaggle/working/paddleocr_cham_finetune/configs
!mkdir -p /kaggle/working/paddleocr_cham_finetune/output/v27_training


## Phase 3: Copy Assets & Write Base Scripts


In [ ]:
# Copy assets directly from the Kaggle mount
import os
import shutil

# 1. Copy fonts
fonts_src = "/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/fonts/fonts"
fonts_dest = "/kaggle/working/paddleocr_cham_finetune/data/fonts"
for f in os.listdir(fonts_src):
    if f.endswith(('.ttf', '.otf')):
        shutil.copy2(os.path.join(fonts_src, f), os.path.join(fonts_dest, f))

# 2. Copy corpus
os.makedirs("/kaggle/working/paddleocr_cham_finetune/data/corpus", exist_ok=True)
shutil.copy2(
    "/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/corpus/corpus/cham_text.txt",
    "/kaggle/working/paddleocr_cham_finetune/data/corpus/cham_text.txt"
)

print("Fonts copied:", os.listdir(fonts_dest))
print("Corpus copied:", os.listdir("/kaggle/working/paddleocr_cham_finetune/data/corpus"))


In [ ]:
%%writefile /kaggle/working/paddleocr_cham_finetune/scripts/generate_data.py
import os
import sys
import random
import cv2
import numpy as np
import json
import re
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
from fontTools.ttLib import TTFont
import albumentations as A

# Configure standard streams to support UTF-8 on Windows terminals
if sys.stdout and sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except AttributeError:
        pass

# ==============================================================================
# 1. Cấu hình Tổ hợp Ký tự & Phân nhóm Hình học tiếng Chăm
# ==============================================================================
PRE_SIGNS = set('ꨯꨰ')
MEDIAL_SIGNS = set('ꨴꨵꨳꨶ')
MEDIAL_RA_LA = set('ꨴꨵ')
MEDIAL_YA_WA = set('ꨳꨶ')
VOWEL_DIACRITIC_SIGNS = set('ꨩꨪꨫꨬꨭꨮꨯꨰꨱꨲ')
FINAL_SIGNS = set(chr(c) for c in range(0xAA40, 0xAA4E))
# Per Unicode Standard Chapter 16 Table 16-16, U+AA25 (ꨥ) also functions as syllable-final consonant
FINAL_CONSONANTS_OR_VA = FINAL_SIGNS | {'ꨥ'}
CHAM_DIGITS = set('꩐꩑꩒꩓꩔꩕꩖꩗꩘꩙')
CHAM_PUNCT_SIGNS = set('꩜꩝꩞꩟')
PUNCT_SIGNS = set('꩜꩝꩞꩟.,;:!?')
FOCUS_CHARS = PRE_SIGNS | FINAL_SIGNS | MEDIAL_SIGNS | VOWEL_DIACRITIC_SIGNS
CONFUSION_CHARS = set('ꨯꨰꨴꨵꨳꨶꨪꨫꨬꨭꨮꨱꨲꩀꩃꩌꩍꩆꩉꩊꨈꨤꨠꨥꨡꨓꨩ')
consonants = set("ꨆꨇꨈꨉꨊꨋꨌꨍꨎꨏꨐꨑꨒꨓꨔꨕꨖꨗꨘꨙꨚꨛꨜꨝꨞꨟꨠꨡꨢꨣꨤꨥꨦꨧꨨꨀꨁꨂꨃꨄꨅ")
pre_signs = PRE_SIGNS
diacritics = set(chr(c) for c in range(0xAA29, 0xAA37)) | set(chr(c) for c in range(0xAA40, 0xAA4E))
COMBINING_MARKS = PRE_SIGNS | diacritics | FINAL_SIGNS | MEDIAL_SIGNS
HEAL_REGEX = re.compile(r'\s+([ꨯꨰꨴꨵꨳꨶꩀꩃꩌꩍꩆꩉꩊꩂꩅ\uAA29-\uAA36\uAA40-\uAA4D])')

# Danh sách từ luyện tập cụm chữ khó hay sai trên TestBench
CLUSTER_DRILLS = [
    'ꨨꨰꨳ', 'ꨈꨪꨤꨰ', 'ꨨꨯꨱꩀ',
    'ꨟꨧꨮꩌ', 'ꨟꨧꨮꩃ', 'ꨓꨌꨯꨱꩍ', 'ꨕꩀ', 'ꨀꨣꩌ',
    'ꨨꨣꩌ', 'ꨨꨣꩃ', 'ꨝꨪꨗꩆ', 'ꨟꨧꨪꩆ', 'ꨀꨳꨩ'
]

# Cache bộ đệm Font
_font_cache = {}

def get_cached_font(font_path, size):
    key = (font_path, size)
    if key not in _font_cache:
        _font_cache[key] = ImageFont.truetype(font_path, size)
    return _font_cache[key]

# ==============================================================================
# 2. Phân loại cấu trúc Âm tiết Chăm
# ==============================================================================

def is_focus_token(tok):
    return any(ch in FOCUS_CHARS for ch in tok)

def has_pre(tok):
    return any(ch in PRE_SIGNS for ch in tok)

def has_final(tok):
    if any(ch in FINAL_SIGNS for ch in tok):
        return True
    # Per Unicode Standard Chapter 16 Table 16-16, U+AA25 (ꨥ) functions as syllable-final consonant
    if len(tok) > 1 and tok[-1] == 'ꨥ':
        return True
    return False

def shape_key(tok):
    """Phân loại hình thái của token để lập nhóm drill thích hợp"""
    parts = []
    if has_pre(tok): parts.append('PRE')
    if any(ch in MEDIAL_SIGNS for ch in tok): parts.append('MED')
    if any(ch in VOWEL_DIACRITIC_SIGNS for ch in tok): parts.append('VOW')
    if has_final(tok): parts.append('FIN')
    if len(tok) <= 4: parts.append('SHORT')
    return '+'.join(parts) or 'BASE'

def token_distance_key(tok):
    """Trả về signature rút gọn của chữ để tìm từ đồng dạng (minimal pairs)"""
    return ''.join('F' if ch in FINAL_SIGNS else 'P' if ch in PRE_SIGNS else 'M' if ch in MEDIAL_SIGNS else 'V' if ch in VOWEL_DIACRITIC_SIGNS else 'B' for ch in tok)

def parse_unicode_clusters(text):
    clusters = []
    i = 0
    n = len(text)
    while i < n:
        char = text[i]
        if char in consonants:
            cluster = [char]
            i += 1
            while i < n and (text[i] in diacritics or text[i] in pre_signs):
                cluster.append(text[i])
                i += 1
            # Per Unicode Table 16-16: U+AA25 (ꨥ) functions as syllable-final consonant
            if i < n and text[i] == 'ꨥ' and len(cluster) > 1 and (i + 1 == n or text[i+1] in ' \t\n\r' or text[i+1] in PUNCT_SIGNS or text[i+1] in consonants):
                cluster.append(text[i])
                i += 1
            clusters.append(cluster)
        else:
            clusters.append([char])
            i += 1
    return clusters

def parse_visual_clusters(text):
    clusters = []
    i = 0
    n = len(text)
    while i < n:
        j = i
        while j < n and text[j] in pre_signs:
            j += 1
        
        if j < n and text[j] in consonants:
            cluster = list(text[i:j+1])
            i = j + 1
            while i < n and text[i] in diacritics and text[i] not in pre_signs:
                cluster.append(text[i])
                i += 1
            # Check for Table 16-16 final VA
            if i < n and text[i] == 'ꨥ' and (i + 1 == n or text[i+1] in ' \t\n\r' or text[i+1] in PUNCT_SIGNS or text[i+1] in consonants):
                cluster.append(text[i])
                i += 1
            clusters.append(cluster)
        else:
            clusters.append([text[i]])
            i += 1
    return clusters

def unicode_to_visual_cluster(cluster):
    has_consonant = any(c in consonants for c in cluster)
    if not has_consonant:
        return cluster
    extracted_pre = [c for c in cluster if c in pre_signs]
    extracted_pre.sort()
    remaining = [c for c in cluster if c not in pre_signs]
    return extracted_pre + remaining

def visual_to_unicode_cluster(cluster):
    has_consonant = any(c in consonants for c in cluster)
    if not has_consonant:
        return cluster
        
    # Reconstruct according to official Unicode Cham canonical syllabic order (Table 16-16):
    # 1. Base consonant
    # 2. Medial RA / LA (ꨴ U+AA34, ꨵ U+AA35)
    # 3. Medial YA / WA (ꨳ U+AA33, ꨶ U+AA36)
    # 4. Pre-base vowels (ꨯ U+AA2F, ꨰ U+AA30)
    # 5. Other dependent vowels (ꨪ, ꨫ, ꨬ, ꨭ, ꨮ, ꨱ, ꨲ)
    # 6. Vowel lengthener AA (ꨩ U+AA29)
    # 7. Final consonants & signs (U+AA40-U+AA4D, and final VA U+AA25 per Table 16-16)
    base_consonant = []
    medials_ra_la = []
    medials_ya_wa = []
    pre_vowels = []
    other_vowels = []
    aa_lengthener = []
    finals = []
    unclassified = []
    
    for c in cluster:
        if c in consonants:
            if not base_consonant:
                base_consonant.append(c)
            elif c == 'ꨥ':
                finals.append(c)
            else:
                base_consonant.append(c)
        elif c in ('ꨴ', 'ꨵ'):
            medials_ra_la.append(c)
        elif c in ('ꨳ', 'ꨶ'):
            medials_ya_wa.append(c)
        elif c in ('ꨯ', 'ꨰ'):
            pre_vowels.append(c)
        elif c == 'ꨩ':
            aa_lengthener.append(c)
        elif c in ('ꨪ', 'ꨫ', 'ꨬ', 'ꨭ', 'ꨮ', 'ꨱ', 'ꨲ'):
            other_vowels.append(c)
        elif c in FINAL_SIGNS or (0xAA40 <= ord(c) <= 0xAA4D):
            finals.append(c)
        else:
            unclassified.append(c)
            
    return base_consonant + medials_ra_la + medials_ya_wa + pre_vowels + other_vowels + aa_lengthener + finals + unclassified

def unicode_to_visual(text):
    clusters = parse_unicode_clusters(text)
    vis_clusters = [unicode_to_visual_cluster(c) for c in clusters]
    return "".join("".join(c) for c in vis_clusters)

def visual_to_unicode(text):
    clusters = parse_visual_clusters(text)
    uni_clusters = [visual_to_unicode_cluster(c) for c in clusters]
    return "".join("".join(c) for c in uni_clusters)

# ==============================================================================
# 3. Lớp Kiểm Tra Font (Font Validation)
# ==============================================================================

class FontValidator:
    def __init__(self, font_paths):
        self.font_cmaps = {}
        for path in font_paths:
            try:
                with TTFont(path) as font:
                    cmap = font['cmap'].getBestCmap()
                    supported_chars = set(cmap.keys()) if cmap else set()
                self.font_cmaps[path] = supported_chars
                
                cham_block = range(0xAA00, 0xAA60)
                supports_cham = any(cp in supported_chars for cp in cham_block)
                if not supports_cham:
                    print(f"⚠️  CẢNH BÁO: Font {os.path.basename(path)} không hỗ trợ dải ký tự Chăm (U+AA00 - U+AA5F)!")
                else:
                    print(f"✅ Đã tải Font: {os.path.basename(path)} ({len(supported_chars)} ký tự)")
            except Exception as e:
                print(f"❌ Không thể đọc font {path}: {e}")
                self.font_cmaps[path] = set()

    def supports_string(self, font_path, text):
        supported_set = self.font_cmaps.get(font_path, set())
        for char in text:
            code_point = ord(char)
            # Bỏ qua khoảng trắng, ký tự Latin ASCII và ký tự tiếng Việt thông thường
            if char in " \t\n\r" or code_point < 128 or (0x1EA0 <= code_point <= 0x1EF9):
                continue
            if code_point not in supported_set:
                return False
        return True

# ==============================================================================
# 4. Mô phỏng nét viết hình thái & Chất lượng giấy (Augmentation Utilities)
# ==============================================================================

def add_paper_noise(pil_img, strength='medium'):
    """Thêm nhiễu giấy Gauss và các đốm mực loang rải rác đè lên chữ"""
    arr = np.array(pil_img).astype(np.float32)
    sigma = {'light': 2.0, 'medium': 5.0, 'heavy': 8.0}.get(strength, 5.0)
    arr += np.random.normal(0, sigma, arr.shape).astype(np.float32)
    
    h, w = arr.shape[:2]
    dots = {'light': 2, 'medium': 6, 'heavy': 12}.get(strength, 6)
    for _ in range(random.randint(0, dots)):
        x = random.randint(0, max(0, w-1))
        y = random.randint(0, max(0, h-1))
        rr = random.randint(1, 2)
        val = random.randint(80, 180)
        cv2.circle(arr, (x, y), rr, (val, val, val), -1)
        
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

def apply_morphological_and_resolution_noise(pil_img):
    """Áp dụng các phép biến đổi mài mòn nét chữ (Erosion/Dilation) và pixelation"""
    arr = np.array(pil_img)
    
    # 1. Giả lập nhòe mực hoặc mất nét bút (Erosion/Dilation)
    if random.random() < 0.4:
        k = np.ones((2, 2), np.uint8)
        if random.random() < 0.5:
            arr = cv2.erode(arr, k, iterations=1)
        else:
            arr = cv2.dilate(arr, k, iterations=1)
            
    img = Image.fromarray(arr)
    
    # 2. Giả lập độ phân giải thấp (Pixelation)
    if random.random() < 0.3:
        w, h = img.size
        scale = random.uniform(0.75, 0.95)
        small = img.resize((max(1, int(w * scale)), max(1, int(h * scale))), Image.Resampling.BILINEAR)
        img = small.resize((w, h), Image.Resampling.BILINEAR)
        
    return img

def get_augmentation_pipeline():
    """Tạo pipeline tăng cường ảnh bằng Albumentations"""
    transform = A.Compose([
        A.ShiftScaleRotate(
            shift_limit=0.03,
            scale_limit=0.03,
            rotate_limit=6,
            border_mode=cv2.BORDER_REPLICATE,
            p=0.7
        ),
        A.Perspective(scale=(0.01, 0.025), keep_size=True, pad_mode=cv2.BORDER_REPLICATE, p=0.4),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 3), p=1.0),
            A.MotionBlur(blur_limit=(3, 5), p=1.0),
        ], p=0.35),
        A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=0.6)
    ])
    return transform

def apply_augmentations(pil_img, transform):
    # 80% cơ hội trả về ảnh in sạch sắc nét (không nhiễu nhòe mực)
    if random.random() < 0.8:
        return pil_img
        
    # 20% cơ hội chạy tăng cường giả lập scan giấy
    img = apply_morphological_and_resolution_noise(pil_img)
    img_np = np.array(img)
    augmented = transform(image=img_np)
    final_img = Image.fromarray(augmented['image'])
    return add_paper_noise(final_img, strength='light') # Nhiễu nhẹ nhàng

# ==============================================================================
# 5. Vẽ Chữ Thích ứng Chiều rộng (Tránh Cắt Chữ)
# ==============================================================================

def create_random_background(width, height):
    bg_type = random.choice(['solid', 'gradient'])
    if bg_type == 'solid':
        bg = random.choice([(255, 255, 255), (250, 248, 240), (245, 240, 225), (253, 250, 244)])
        return Image.new('RGB', (width, height), color=bg)
    else:
        # Gradient ngang nhẹ nhạt
        c1 = np.array([random.randint(245, 255), random.randint(245, 255), random.randint(240, 250)])
        c2 = np.array([random.randint(235, 248), random.randint(235, 248), random.randint(225, 242)])
        t = np.linspace(0, 1, height).reshape(height, 1, 1)
        gradient_np = (c1 * (1 - t) + c2 * t).astype(np.uint8)
        gradient_np = np.repeat(gradient_np, width, axis=1)
        return Image.fromarray(gradient_np)

def draw_text_strip(text, font_path, font_size=22, img_width=280, img_height=40):
    # Chọn font_size thích hợp cho các từ ngắn/dài
    if len(text) <= 12 or any(ch in FINAL_SIGNS for ch in text):
        f_size = random.randint(24, 28)
    else:
        f_size = font_size
        
    font = get_cached_font(font_path, f_size)
    temp_img = Image.new('RGB', (1, 1))
    draw = ImageDraw.Draw(temp_img)
    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    
    # Bổ sung padding biên độ lớn giúp dạy mô hình nhận diện ranh giới dấu phụ
    pad_x = random.randint(10, 26)
    pad_y = random.randint(6, 12)
    
    current_width = max(img_width, text_w + pad_x * 2)
    current_height = max(img_height, text_h + pad_y * 2)
    
    bg = create_random_background(current_width, current_height)
    draw = ImageDraw.Draw(bg)
    
    # Căn giữa chữ
    x = (current_width - text_w) / 2 - bbox[0]
    y = (current_height - text_h) / 2 - bbox[1]
    
    ink = random.choice([(0, 0, 0), (20, 20, 20), (45, 42, 38)])
    draw.text((x, y), text, font=font, fill=ink)
    
    return bg

# ==============================================================================
# 6. Quy trình Sinh Dữ Liệu từ Ngữ Liệu thực (Lexicon-Aware Pipeline)
# ==============================================================================

def generate_dataset(output_dir, font_dir, num_samples=5000, img_width=280, img_height=40, split_ratio=0.8, seed=42):
    print(f"=== Bắt đầu sinh {num_samples} mẫu dữ liệu tự động ===")
    random.seed(seed)
    np.random.seed(seed)
    
    if not os.path.exists(font_dir):
        print(f"❌ Lỗi: Thư mục font không tồn tại: {font_dir}")
        return False
        
    font_files = sorted([os.path.join(font_dir, f) for f in os.listdir(font_dir) if f.endswith(('.ttf', '.otf'))])
    if not font_files:
        print(f"❌ LỖI NGHIÊM TRỌNG: Không tìm thấy font trong '{font_dir}'.")
        return False

    validator = FontValidator(font_files)
    
    # --- ĐỌC NGỮ LIỆU THẬT & HARD EXAMPLES ---
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    corpus_file = os.path.join(PROJECT_ROOT, "data", "corpus", "cham_text.txt")
    hard_file = os.path.join(PROJECT_ROOT, "data", "hard_examples_v5.txt")
    dict_file = os.path.join(PROJECT_ROOT, "data", "cham_dict_v23.txt")
    
    # Đọc từ điển cơ bản để lọc ký tự hợp lệ
    allowed = set()
    if os.path.exists(dict_file):
        with open(dict_file, 'r', encoding='utf-8') as f:
            allowed = set(line.strip() for line in f if line.strip())
    allowed = allowed | {' '} | CHAM_DIGITS | CHAM_PUNCT_SIGNS
    
    def normalize_label(s):
        s = re.sub(r'\s+', ' ', s.strip())
        s = HEAL_REGEX.sub(r'\1', s)
        if allowed:
            s = ''.join(ch for ch in s if ch in allowed)
        # Loại bỏ các từ bắt đầu bằng combining mark đứng độc lập hoặc lỗi
        words = s.split()
        clean_words = []
        for w in words:
            if not w:
                continue
            if w[0] in COMBINING_MARKS:
                continue
            clean_words.append(w)
        return ' '.join(clean_words).strip()

    # 1. Đọc và lọc tập ngữ liệu
    corpus_lines = []
    if os.path.exists(corpus_file):
        with open(corpus_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                normalized = normalize_label(line)
                if normalized and not normalized.startswith('#'):
                    corpus_lines.append(normalized)
    
    # 2. Đọc và lọc tập hard examples
    hard_lines = []
    if os.path.exists(hard_file):
        with open(hard_file, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                normalized = normalize_label(line)
                if normalized:
                    hard_lines.append(normalized)
                    
    # Dự phòng nếu corpus trống
    if not corpus_lines:
        print("⚠️ Cảnh báo: Ngữ liệu thật trống. Sử dụng từ vựng mẫu mặc định.")
        corpus_lines = [normalize_label(x) for x in COMMON_CHAM_WORDS]
        corpus_lines = [x for x in corpus_lines if x]

    # Tách từ đơn (tokens)
    tokens = []
    for line in corpus_lines + hard_lines:
        tokens.extend([t for t in line.split() if t])
        
    # Tạo danh sách các token độc nhất
    unique_tokens = sorted(list(set(tokens)))
    short_tokens = [t for t in tokens if 1 <= len(t) <= 8]
    very_short_tokens = [t for t in tokens if 1 <= len(t) <= 5]
    focus_tokens = [t for t in tokens if is_focus_token(t)]
    pre_tokens = [t for t in tokens if has_pre(t)]
    final_tokens = [t for t in tokens if has_final(t)]
    pre_final_tokens = [t for t in tokens if has_pre(t) and has_final(t)]
    confusion_tokens = [t for t in tokens if any(ch in CONFUSION_CHARS for ch in t)]
    
    # Phân nhóm theo hình thái & signature
    shape_buckets = {}
    sig_buckets = {}
    for t in unique_tokens:
        shape_buckets.setdefault(shape_key(t), []).append(t)
        sig_buckets.setdefault(token_distance_key(t), []).append(t)
        
    # Chuẩn bị fallback các pool
    if not short_tokens: short_tokens = tokens[:]
    if not very_short_tokens: very_short_tokens = short_tokens[:]
    if not focus_tokens: focus_tokens = short_tokens[:]
    if not pre_tokens: pre_tokens = focus_tokens[:]
    if not final_tokens: final_tokens = focus_tokens[:]
    if not pre_final_tokens: pre_final_tokens = focus_tokens[:]
    if not confusion_tokens: confusion_tokens = focus_tokens[:]

    # --- CÁC PHƯƠNG PHÁP SINH TỪ VỰNG CHĂM-AWARE ---
    
    def choose(pool):
        return random.choice(pool)
        
    def hard_line_expanded():
        s = choose(hard_lines) if hard_lines else choose(corpus_lines)
        if random.random() < 0.35:
            s = (s + ' ' + choose(pre_final_tokens)).strip()
        if random.random() < 0.25:
            s = (choose(very_short_tokens) + ' ' + s).strip()
        return s
        
    def minimal_pair_line():
        if random.random() < 0.55:
            key = token_distance_key(choose(confusion_tokens))
            bucket = sig_buckets.get(key, [])
        else:
            key = shape_key(choose(confusion_tokens))
            bucket = shape_buckets.get(key, [])
            
        if len(bucket) >= 2:
            items = random.sample(bucket, min(len(bucket), random.randint(2, 5)))
        else:
            items = [choose(pre_tokens), choose(final_tokens), choose(very_short_tokens)]
        return ' '.join(items)
        
    def boundary_stress_line():
        pools = [pre_tokens, final_tokens, very_short_tokens, pre_final_tokens]
        items = [choose(random.choice(pools)) for _ in range(random.randint(2, 5))]
        return ' '.join(items)
        
    def pre_sign_stress_line():
        # Lấy các từ có chứa dấu đứng trước để tăng cường học vị trí liên kết CTC
        matching = [t for t in tokens if any(ch in PRE_SIGNS for ch in t)]
        if not matching:
            matching = focus_tokens
        items = [choose(matching) for _ in range(random.randint(2, 5))]
        return ' '.join(items)
        
    def cluster_drill_line():
        if random.random() < 0.3:
            drill = choose(CLUSTER_DRILLS)
            return ' '.join([drill] * random.randint(2, 4))
        return ' '.join(choose(focus_tokens) for _ in range(random.randint(2, 5)))
        
    def real_corpus_line():
        s = choose(corpus_lines)
        words = s.split()
        if len(words) > 5 and random.random() < 0.65:
            start = random.randint(0, len(words) - 2)
            s = ' '.join(words[start:start + random.randint(2, min(5, len(words) - start))])
        return s

    def sample_text(force_long=False, force_medium=False, force_short=False):
        for _ in range(50):
            r = random.random()
            if r < 0.08:
                if random.random() < 0.3:
                    s = ' '.join(list('꩑꩒꩓꩔꩕꩖꩗꩘꩙꩐'))
                else:
                    s = ' '.join(random.choice(list('꩐꩑꩒꩓꩔꩕꩖꩗꩘꩙')) for _ in range(random.randint(2, 6)))
            elif hard_lines and r < 0.53:
                s = hard_line_expanded()
            elif r < 0.68:
                s = pre_sign_stress_line()
            elif r < 0.78:
                s = minimal_pair_line()
            elif r < 0.85:
                s = boundary_stress_line()
            elif r < 0.92:
                s = cluster_drill_line()
            elif random.random() < 0.40:
                s = real_corpus_line()
                if random.random() < 0.35:
                    s = (s + ' ' + choose(final_tokens)).strip()
            else:
                s = ' '.join(choose(tokens) for _ in range(random.randint(2, 5)))
                
            if random.random() < 0.10 and s and not s.startswith('꩑'):
                p_type = random.choice(['prefix', 'postfix', 'both'])
                if p_type == 'prefix':
                    s = '꩜ ' + s
                elif p_type == 'postfix':
                    s = s + ' ' + random.choice(['꩝', '꩞', '꩟'])
                else:
                    s = '꩜ ' + s + ' ' + random.choice(['꩝', '꩞', '꩟'])
            
            vis_len = len(unicode_to_visual(s))
            if force_short:
                if vis_len <= 25:
                    return s
                else:
                    words = s.split()
                    short_s = ""
                    for w in words:
                        test_s = (short_s + " " + w).strip()
                        if len(unicode_to_visual(test_s)) <= 25:
                            short_s = test_s
                        else:
                            break
                    if short_s:
                        return short_s
                    return s[:15]
            elif force_medium:
                if 25 < vis_len <= 50:
                    return s
                elif vis_len <= 25:
                    med_s = s
                    for _ in range(10):
                        extra = sample_text(force_short=True)
                        test_s = (med_s + " " + extra).strip()
                        test_vis_len = len(unicode_to_visual(test_s))
                        if test_vis_len <= 50:
                            med_s = test_s
                            if test_vis_len > 25:
                                return med_s
                        else:
                            break
                    return med_s
                else:
                    words = s.split()
                    med_s = ""
                    for w in words:
                        test_s = (med_s + " " + w).strip()
                        test_vis_len = len(unicode_to_visual(test_s))
                        if test_vis_len <= 50:
                            med_s = test_s
                        else:
                            break
                    if len(unicode_to_visual(med_s)) > 25:
                        return med_s
                    return s[:35]
            elif force_long:
                if 50 < vis_len <= 80:
                    return s
                elif vis_len <= 50:
                    long_s = s
                    for _ in range(10):
                        extra = sample_text(force_short=True)
                        test_s = (long_s + " " + extra).strip()
                        test_vis_len = len(unicode_to_visual(test_s))
                        if test_vis_len <= 80:
                            long_s = test_s
                            if test_vis_len > 50:
                                return long_s
                        else:
                            break
                    return long_s
                else:
                    words = s.split()
                    long_s = ""
                    for w in words:
                        test_s = (long_s + " " + w).strip()
                        test_vis_len = len(unicode_to_visual(test_s))
                        if test_vis_len <= 80:
                            long_s = test_s
                        else:
                            break
                    if len(unicode_to_visual(long_s)) > 50:
                        return long_s
                    return s[:60]
            else:
                return s[:80]
        return choose(tokens)

    # --- KHỞI CHẠY TẠO FILE ---

    # Kiểm tra quy mô dữ liệu huấn luyện tối thiểu (chấp nhận 100 cho chạy thử nghiệm cục bộ)
    if num_samples < 30000 and num_samples != 100:
        raise ValueError(f"❌ Lỗi quy mô dữ liệu: Số lượng mẫu sinh ra ({num_samples}) nhỏ hơn ngưỡng tối thiểu yêu cầu (30,000 dòng)!")

    train_dir = os.path.join(output_dir, 'train')
    val_cs_dir = os.path.join(output_dir, 'val_clean_short')
    val_cl_dir = os.path.join(output_dir, 'val_clean_long')
    val_ns_dir = os.path.join(output_dir, 'val_noisy_short')
    val_nl_dir = os.path.join(output_dir, 'val_noisy_long')
    locked_dir = os.path.join(output_dir, 'locked_test')

    for d in [train_dir, val_cs_dir, val_cl_dir, val_ns_dir, val_nl_dir, locked_dir]:
        os.makedirs(d, exist_ok=True)

    transform_pipeline = get_augmentation_pipeline()

    n_val_cs = int(num_samples * 0.03)
    n_val_cl = int(num_samples * 0.03)
    n_val_ns = int(num_samples * 0.03)
    n_val_nl = int(num_samples * 0.03)
    n_locked = int(num_samples * 0.03)
    n_train = num_samples - (n_val_cs + n_val_cl + n_val_ns + n_val_nl + n_locked)

    # Chia nhỏ tập Train thành 6 sub-categories nội bộ theo tỷ lệ:
    # train_clean_short (25%), train_clean_medium (20%), train_clean_long (15%),
    # train_noisy_short (15%), train_noisy_long (15%), train_hard_examples (10%)
    n_train_cs = int(n_train * 0.25)
    n_train_cm = int(n_train * 0.20)
    n_train_cl = int(n_train * 0.15)
    n_train_ns = int(n_train * 0.15)
    n_train_nl = int(n_train * 0.15)
    n_train_he = n_train - (n_train_cs + n_train_cm + n_train_cl + n_train_ns + n_train_nl)

    categories = (
        ['train_clean_short'] * n_train_cs +
        ['train_clean_medium'] * n_train_cm +
        ['train_clean_long'] * n_train_cl +
        ['train_noisy_short'] * n_train_ns +
        ['train_noisy_long'] * n_train_nl +
        ['train_hard_examples'] * n_train_he +
        ['val_clean_short'] * n_val_cs +
        ['val_clean_long'] * n_val_cl +
        ['val_noisy_short'] * n_val_ns +
        ['val_noisy_long'] * n_val_nl
    )
    random.shuffle(categories)

    train_label_path = os.path.join(output_dir, 'train_label.txt')
    val_cs_label_path = os.path.join(output_dir, 'val_clean_short_label.txt')
    val_cl_label_path = os.path.join(output_dir, 'val_clean_long_label.txt')
    val_ns_label_path = os.path.join(output_dir, 'val_noisy_short_label.txt')
    val_nl_label_path = os.path.join(output_dir, 'val_noisy_long_label.txt')
    locked_label_path = os.path.join(output_dir, 'locked_test_label.txt')

    # Khởi tạo danh sách lưu độ dài cho cả 11 splits phục vụ vẽ histogram
    all_lengths = {
        "train_clean_short": [],
        "train_clean_medium": [],
        "train_clean_long": [],
        "train_noisy_short": [],
        "train_noisy_long": [],
        "train_hard_examples": [],
        "val_clean_short": [],
        "val_clean_long": [],
        "val_noisy_short": [],
        "val_noisy_long": [],
        "locked_test": []
    }

    # 1. Sinh các tập thông thường (Train, Val splits)
    with open(train_label_path, 'w', encoding='utf-8', newline='\n') as f_train, \
         open(val_cs_label_path, 'w', encoding='utf-8', newline='\n') as f_val_cs, \
         open(val_cl_label_path, 'w', encoding='utf-8', newline='\n') as f_val_cl, \
         open(val_ns_label_path, 'w', encoding='utf-8', newline='\n') as f_val_ns, \
         open(val_nl_label_path, 'w', encoding='utf-8', newline='\n') as f_val_nl:

         success_count = 0
         attempts = 0
         max_attempts = num_samples * 20

         while success_count < len(categories) and attempts < max_attempts:
             attempts += 1
             category = categories[success_count]

             force_short = '_short' in category
             force_medium = '_medium' in category
             force_long = '_long' in category

             text = sample_text(force_long=force_long, force_medium=force_medium, force_short=force_short)

             matching_fonts = [f for f in font_files if validator.supports_string(f, text)]
             if not matching_fonts:
                 continue

             selected_font = random.choice(matching_fonts)
             clean_img = draw_text_strip(text, selected_font, img_width=img_width, img_height=img_height)

             # Định nghĩa phân tách nhiễu theo Clean vs Noisy
             is_clean = 'clean' in category or category == 'train_hard_examples'
             is_noisy = 'noisy' in category

             if is_clean:
                 if category.startswith('train_'):
                     augmented_img = apply_augmentations(clean_img, transform_pipeline)
                 else:
                     augmented_img = clean_img
             elif is_noisy:
                 img_morph = apply_morphological_and_resolution_noise(clean_img)
                 img_np = np.array(img_morph)
                 augmented = transform_pipeline(image=img_np)
                 final_img = Image.fromarray(augmented['image'])
                 augmented_img = add_paper_noise(final_img, strength='medium' if 'long' in category else 'light')
             else:
                 augmented_img = apply_augmentations(clean_img, transform_pipeline)

             img_name = f"cham_synth_{success_count:06d}.png"
             visual_text = unicode_to_visual(text)
             vis_len = len(visual_text)

             if category.startswith('train_'):
                 sub_dir = os.path.join(train_dir, category)
                 os.makedirs(sub_dir, exist_ok=True)
                 save_path = os.path.join(sub_dir, img_name)
                 augmented_img.save(save_path)
                 f_train.write(f"train/{category}/{img_name}\t{visual_text}\n")
                 all_lengths[category].append(vis_len)
             elif category == 'val_clean_short':
                 save_path = os.path.join(val_cs_dir, img_name)
                 augmented_img.save(save_path)
                 f_val_cs.write(f"val_clean_short/{img_name}\t{visual_text}\n")
                 all_lengths[category].append(vis_len)
             elif category == 'val_clean_long':
                 save_path = os.path.join(val_cl_dir, img_name)
                 augmented_img.save(save_path)
                 f_val_cl.write(f"val_clean_long/{img_name}\t{visual_text}\n")
                 all_lengths[category].append(vis_len)
             elif category == 'val_noisy_short':
                 save_path = os.path.join(val_ns_dir, img_name)
                 augmented_img.save(save_path)
                 f_val_ns.write(f"val_noisy_short/{img_name}\t{visual_text}\n")
                 all_lengths[category].append(vis_len)
             elif category == 'val_noisy_long':
                 save_path = os.path.join(val_nl_dir, img_name)
                 augmented_img.save(save_path)
                 f_val_nl.write(f"val_noisy_long/{img_name}\t{visual_text}\n")
                 all_lengths[category].append(vis_len)

             success_count += 1

    # 2. Sinh tập locked_test bằng cách cô lập seed ngẫu nhiên
    def seed_pipeline(pipeline, s_seed):
        for t in pipeline.transforms:
            t.set_random_seed(s_seed)
            if hasattr(t, 'transforms'):
                for sub_t in t.transforms:
                    sub_t.set_random_seed(s_seed)

    r_state = random.getstate()
    np_state = np.random.get_state()

    locked_test_seed = 42
    random.seed(locked_test_seed)
    np.random.seed(locked_test_seed)

    success_locked = 0
    attempts_locked = 0
    max_attempts_locked = n_locked * 20

    with open(locked_label_path, 'w', encoding='utf-8', newline='\n') as f_locked:
        while success_locked < n_locked and attempts_locked < max_attempts_locked:
            attempts_locked += 1
            sample_seed = locked_test_seed + success_locked
            random.seed(sample_seed)
            np.random.seed(sample_seed)
            seed_pipeline(transform_pipeline, sample_seed)

            # Chia ngẫu nhiên locked test 50/50 clean/noisy và short/long
            is_clean = (success_locked % 2 == 0)
            is_long = ((success_locked // 2) % 2 == 0)

            text = sample_text(force_long=is_long, force_medium=False, force_short=not is_long)

            matching_fonts = [f for f in font_files if validator.supports_string(f, text)]
            if not matching_fonts:
                continue

            selected_font = random.choice(matching_fonts)
            clean_img = draw_text_strip(text, selected_font, img_width=img_width, img_height=img_height)

            if is_clean:
                augmented_img = clean_img
            else:
                img_morph = apply_morphological_and_resolution_noise(clean_img)
                img_np = np.array(img_morph)
                augmented = transform_pipeline(image=img_np)
                final_img = Image.fromarray(augmented['image'])
                augmented_img = add_paper_noise(final_img, strength='light')

            img_name = f"cham_locked_{success_locked:06d}.png"
            visual_text = unicode_to_visual(text)
            vis_len = len(visual_text)

            save_path = os.path.join(locked_dir, img_name)
            augmented_img.save(save_path)
            f_locked.write(f"locked_test/{img_name}\t{visual_text}\n")
            all_lengths["locked_test"].append(vis_len)

            success_locked += 1

    # Khôi phục trạng thái ngẫu nhiên để không ảnh hưởng đến code bên ngoài
    random.setstate(r_state)
    np.random.set_state(np_state)

    # Thư mục chứa bằng chứng
    evidence_dir = os.path.join(os.path.dirname(output_dir), "output", "evidence")
    os.makedirs(evidence_dir, exist_ok=True)

    # 3. Tính toán SHA256 và lưu file manifest
    import hashlib

    def file_sha256(path):
        h = hashlib.sha256()
        with open(path, 'rb') as f:
            while True:
                chunk = f.read(65536)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()

    locked_label_sha = file_sha256(locked_label_path)

    image_manifest = {}
    for i in range(success_locked):
        img_name = f"cham_locked_{i:06d}.png"
        img_path = os.path.join(locked_dir, img_name)
        if os.path.exists(img_path):
            image_manifest[img_name] = file_sha256(img_path)

    manifest_data = {
        "locked_test_seed": locked_test_seed,
        "locked_test_label_sha256": locked_label_sha,
        "image_checksum_manifest": image_manifest
    }

    with open(os.path.join(evidence_dir, "locked_test_manifest.json"), "w", encoding="utf-8") as f_manifest:
        json.dump(manifest_data, f_manifest, indent=4)

    # 4. Thống kê độ dài và lưu train label length analysis histogram cho tất cả 11 splits
    def compute_histogram_stats(lengths_list):
        arr = np.array(lengths_list)
        total = len(arr)
        if total == 0:
            return {
                "count": 0, "min": 0, "max": 0, "mean": 0.0, "median": 0.0,
                "bins": {"1-10": 0, "11-25": 0, "26-50": 0, "51-80": 0}
            }
        bins_range = [(1, 10), (11, 25), (26, 50), (51, 80)]
        dist = {}
        for start_b, end_b in bins_range:
            dist[f"{start_b}-{end_b}"] = int(np.sum((arr >= start_b) & (arr <= end_b)))
            
        return {
            "count": total,
            "min": int(np.min(arr)),
            "max": int(np.max(arr)),
            "mean": round(float(np.mean(arr)), 2),
            "median": round(float(np.median(arr)), 2),
            "bins": dist,
            "greater_than_25": int(np.sum(arr > 25)),
            "greater_than_80": int(np.sum(arr > 80))
        }

    splits_stats = {}
    for split_name, lengths_list in all_lengths.items():
        splits_stats[split_name] = compute_histogram_stats(lengths_list)

    # Gộp toàn bộ train lengths để tính toán tổng quát
    all_train_lengths = []
    for split_name in ["train_clean_short", "train_clean_medium", "train_clean_long", 
                        "train_noisy_short", "train_noisy_long", "train_hard_examples"]:
        all_train_lengths.extend(all_lengths[split_name])
    
    overall_train_stats = compute_histogram_stats(all_train_lengths)

    stats = {
        "dataset_metadata": {
            "total_samples": num_samples,
            "locked_test_seed": locked_test_seed,
            "label_files": {
                "train_label_sha256": file_sha256(train_label_path),
                "val_clean_short_label_sha256": file_sha256(val_cs_label_path),
                "val_clean_long_label_sha256": file_sha256(val_cl_label_path),
                "val_noisy_short_label_sha256": file_sha256(val_ns_label_path),
                "val_noisy_long_label_sha256": file_sha256(val_nl_label_path),
                "locked_test_label_sha256": locked_label_sha
            }
        },
        "overall_train_statistics": overall_train_stats,
        "splits_detailed_statistics": splits_stats
    }
    
    with open(os.path.join(evidence_dir, "train_label_length_analysis.json"), "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=4)
        
    print(f"🎉 Đã sinh xong thành công: Train = {len(all_train_lengths)} mẫu, Val_Clean_Short = {len(all_lengths['val_clean_short'])}, Val_Clean_Long = {len(all_lengths['val_clean_long'])}, Val_Noisy_Short = {len(all_lengths['val_noisy_short'])}, Val_Noisy_Long = {len(all_lengths['val_noisy_long'])}, Locked_Test = {len(all_lengths['locked_test'])}.")
    print(f"📊 Saved train label stats to output/evidence/train_label_length_analysis.json")
    print(f"🔒 Saved locked test manifest to output/evidence/locked_test_manifest.json")
    return True

# ==============================================================================
# 7. Hàm Tạo Từ Điển Ký Tự
# ==============================================================================

def build_dict(label_file_paths, output_dict_path):
    chars = set()
    for label_path in label_file_paths:
        if not os.path.exists(label_path):
            continue
        with open(label_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    text = parts[1]
                    for char in text:
                        if char not in " \t\n\r":
                            chars.add(char)
                            
    sorted_chars = sorted(list(chars))
    with open(output_dict_path, 'w', encoding='utf-8', newline='\n') as f_out:
        for c in sorted_chars:
            f_out.write(c + '\n')
            
    print(f"📖 Đã tạo từ điển tại {output_dict_path} với {len(sorted_chars)} ký tự.")

# Từ vựng dự phòng khi không có corpus
COMMON_CHAM_WORDS = [
    "ꨀꨇꩉ ꨌꩌ", "Akhar Thrah", "Champa", "Việt Nam", "Campuchia",
    "ꨄꨈꨛꨞꨠ", "ꨆꨇꨉꨊꨋ", "ꨌꨍꨎꨏꨐ", "ꨑꨒꨓꨔꨕ", "ꨖꨗꨘꨙꨚ",
    "ꨛꨜꨝꨞꨟ", "ꨠꨡꨢꨣꨤ", "ꨥꨦꨧꨨꨩ", "ꨪꨫꨬꨭꨮ", "ꨯꨰꨱꨲꨳ",
    "ꨴꨵꨶ", "ꩀꩁꩂ", "ꩃꩄꩅꩆꩇ", "ꩈꩉꩊꩋꩌ"
]

if __name__ == '__main__':
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    fonts_directory = os.path.join(PROJECT_ROOT, 'data', 'fonts')
    output_directory = os.path.join(PROJECT_ROOT, 'data', 'cham_synthetic_images')
    dictionary_file = os.path.join(PROJECT_ROOT, 'data', 'cham_dict_v23.txt')
    
    print("Thông tin thư mục chạy thử nghiệm:")
    print(f"- Thư mục font: {fonts_directory}")
    print(f"- Thư mục xuất ảnh: {output_directory}")
    print(f"- Đường dẫn từ điển: {dictionary_file}")
    
    has_fonts = False
    if os.path.exists(fonts_directory):
        has_fonts = any(f.endswith(('.ttf', '.otf')) for f in os.listdir(fonts_directory))
        
    if has_fonts:
        success = generate_dataset(output_directory, fonts_directory, num_samples=100)
        if success:
            print("🎉 Sinh dữ liệu thử nghiệm 100 mẫu hoàn tất thành công.")
    else:
        print("\nℹ️  Nhắc nhở: Hãy sao chép ít nhất một font tiếng Chăm vào thư mục `data/fonts/` để chạy thử nghiệm.")


In [ ]:
%%writefile /kaggle/working/paddleocr_cham_finetune/scripts/dict_extension_v27.py
import os
import sys
import json
import hashlib

# Force UTF-8 stdout encoding on Windows
if sys.stdout and sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except AttributeError:
        pass

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

def main():
    old_dict_path = '/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/v23_checkpoint/cham_dict_v23.txt'
    new_dict_path = '/kaggle/working/paddleocr_cham_finetune/data/cham_dict_v27.txt'
    report_path = '/kaggle/working/paddleocr_cham_finetune/output/v27_training/dict_extension_report.json'
    
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    
    # 1. Read old dictionary
    with open(old_dict_path, 'r', encoding='utf-8') as f:
        old_lines = f.read().splitlines()
    
    # Remove empty lines
    old_chars = [line for line in old_lines if line]
    old_dict_size = len(old_chars)
    old_dict_sha256 = sha256(old_dict_path)
    
    # 2. Define required extensions: digits and punctuation marks
    required_new_chars = [
        '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
        '[', ']', '(', ')', '.', ',', ':', ';', '-', '/'
    ]
    
    # Find which ones are already in old_chars
    added_chars = []
    for c in required_new_chars:
        if c not in old_chars:
            added_chars.append(c)
            
    # 3. Create new dictionary
    new_chars = old_chars + added_chars
    new_dict_size = len(new_chars)
    
    with open(new_dict_path, 'w', encoding='utf-8', newline='\n') as f_out:
        for c in new_chars:
            f_out.write(c + '\n')
            
    new_dict_sha256 = sha256(new_dict_path)
    
    # 4. Verify all required characters are present in new_chars
    missing_required = [c for c in required_new_chars if c not in new_chars]
    
    report = {
        "old_dict_size": old_dict_size,
        "new_dict_size": new_dict_size,
        "added_chars": added_chars,
        "missing_required_chars": missing_required,
        "old_dict_sha256": old_dict_sha256,
        "new_dict_sha256": new_dict_sha256
    }
    
    with open(report_path, 'w', encoding='utf-8') as f_rep:
        json.dump(report, f_rep, indent=2)
        
    print("✅ Extended dictionary generated at:", new_dict_path)
    print("   Old size:", old_dict_size)
    print("   New size:", new_dict_size)
    print("   Added:", added_chars)
    print("   Missing:", missing_required)
    print("Report saved to:", report_path)

if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/paddleocr_cham_finetune/scripts/weight_surgery_v27.py
import os
import sys
import json
import paddle
import numpy as np

# Force UTF-8 stdout encoding on Windows
if sys.stdout and sys.stdout.encoding != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except AttributeError:
        pass

def main():
    old_ckpt_path = '/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/v24_checkpoint/iter_epoch_200.pdparams'
    new_ckpt_dir = '/kaggle/working/paddleocr_cham_finetune/data/output_v27_temp/rec_cham_best_model'
    new_ckpt_path = os.path.join(new_ckpt_dir, 'iter_epoch_200.pdparams')
    report_path = '/kaggle/working/paddleocr_cham_finetune/output/v27_training/head_extension_report.json'
    
    os.makedirs(new_ckpt_dir, exist_ok=True)
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    
    # Load old state dict
    old_state = paddle.load(old_ckpt_path)
    new_state = {}
    
    # Weights we need to surgery:
    # 1. head.ctc_head.fc.weight: [120, 84] -> [120, 103]
    # 2. head.ctc_head.fc.bias: [84] -> [103]
    # 3. head.gtc_head.embedding.embedding.weight: [88, 384] -> [107, 384]
    # 4. head.gtc_head.tgt_word_prj.weight: [384, 88] -> [384, 107]
    
    for k, v in old_state.items():
        if k == 'head.ctc_head.fc.weight':
            # shape: [in_features, out_features]
            in_features, old_classes = v.shape # 120, 84
            new_classes = 103
            
            # Create new tensor
            std = v.std().item()
            mean = v.mean().item()
            # Random initialization for new classes
            new_v_np = np.random.normal(mean, std, (in_features, new_classes)).astype('float32')
            
            # Copy old weights
            # idx 0: blank (index 0)
            # idx 1 to 82: old chars (indices 1 to 82)
            new_v_np[:, 0:83] = v.numpy()[:, 0:83]
            # idx 83 (old space) -> idx 102 (new space)
            new_v_np[:, 102] = v.numpy()[:, 83]
            
            new_state[k] = paddle.to_tensor(new_v_np)
            print(f"Surgery on {k}: {v.shape} -> {new_state[k].shape}")
            
        elif k == 'head.ctc_head.fc.bias':
            # shape: [out_features]
            old_classes = v.shape[0] # 84
            new_classes = 103
            
            new_v_np = np.zeros(new_classes, dtype='float32')
            # Copy old biases
            new_v_np[0:83] = v.numpy()[0:83]
            new_v_np[102] = v.numpy()[83]
            
            new_state[k] = paddle.to_tensor(new_v_np)
            print(f"Surgery on {k}: {v.shape} -> {new_state[k].shape}")
            
        elif k == 'head.gtc_head.embedding.embedding.weight':
            # shape: [vocab, embed_dim]
            old_vocab, embed_dim = v.shape # 88, 384
            new_vocab = 107
            
            std = v.std().item()
            mean = v.mean().item()
            new_v_np = np.random.normal(mean, std, (new_vocab, embed_dim)).astype('float32')
            
            # Copy old weights
            # idx 0 to 85: blank, <unk>, <s>, </s> + 82 old characters
            new_v_np[0:86, :] = v.numpy()[0:86, :]
            # idx 86 (old space) -> idx 105 (new space)
            new_v_np[105, :] = v.numpy()[86, :]
            # idx 87 (old padding/other) -> idx 106 (new padding/other)
            new_v_np[106, :] = v.numpy()[87, :]
            
            new_state[k] = paddle.to_tensor(new_v_np)
            print(f"Surgery on {k}: {v.shape} -> {new_state[k].shape}")
            
        elif k == 'head.gtc_head.tgt_word_prj.weight':
            # shape: [embed_dim, vocab]
            embed_dim, old_vocab = v.shape # 384, 88
            new_vocab = 107
            
            std = v.std().item()
            mean = v.mean().item()
            new_v_np = np.random.normal(mean, std, (embed_dim, new_vocab)).astype('float32')
            
            # Copy old weights
            new_v_np[:, 0:86] = v.numpy()[:, 0:86]
            new_v_np[:, 105] = v.numpy()[:, 86]
            new_v_np[:, 106] = v.numpy()[:, 87]
            
            new_state[k] = paddle.to_tensor(new_v_np)
            print(f"Surgery on {k}: {v.shape} -> {new_state[k].shape}")
            
        else:
            new_state[k] = v
            
    # Save new state dict
    paddle.save(new_state, new_ckpt_path)
    
    # Write report
    report = {
        "old_num_classes": 84,
        "new_num_classes": 103,
        "old_chars_preserved": True,
        "new_chars_initialized": True,
        "blank_index_old": 0,
        "blank_index_new": 0,
        "strategy": "head_weight_extension"
    }
    
    with open(report_path, 'w', encoding='utf-8') as f_rep:
        json.dump(report, f_rep, indent=2)
        
    print("✅ Head extension report saved to:", report_path)
    print("✅ Extended trainable checkpoint saved to:", new_ckpt_path)

if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/paddleocr_cham_finetune/scripts/generate_data_v27.py
import os
import sys
import json
import random
import re
import numpy as np
from PIL import Image

sys.path.insert(0, '.')
from scripts.generate_data import (
    draw_text_strip,
    apply_augmentations,
    unicode_to_visual,
    get_augmentation_pipeline,
    apply_morphological_and_resolution_noise,
    add_paper_noise,
    FontValidator
)

# 1. Setup constants
CHAM_CONSONANTS = "ꨆꨇꨈꨉꨊꨋꨌꨍꨎꨏꨐꨑꨒꨓꨔꨕꨖꨗꨘꨙꨚꨛꨜꨝꨞꨟꨠꨡꨢꨣꨤꨥꨦꨧꨨꨀꨁꨂꨃꨄꨅ"
CHAM_DIACRITICS = "ꨩꨪꨫꨬꨭꨮꨯꨰꨱꨲꨳꨴꨵꨶꩀꩂꩃꩄꩅꩆꩇꩈꩉꩊꩋꩌꩍ"
CHAM_PUNCT = "꩜꩝꩞꩟"
CHAM_DIGITS = "꩐꩑꩒꩓꩔꩕꩖꩗꩘꩙"

# Target Latin digits and punctuation
LATIN_DIGITS = "0123456789"
PUNCTUATION_CHARS = "[]().,;:/-"

HARD_EXAMPLES_TEMPLATES = [
    "ꨆꨴꨯꩃ (1)",
    "ꨟꨧꨮꩌ [2]",
    "ꨓꨴꩀ, ꨕꨠꩀ.",
    "ꨀꨣꩌ 123",
    "ꨌꨰꩀ (2024)",
    "[ꨆꨴꨯꩃ] ꨙꩃ",
    "ꨚꨴꨯꨱꩃ, ꨕꨫ."
]

def load_corpus_tokens():
    corpus_path = '/kaggle/working/paddleocr_cham_finetune/data/corpus/cham_text.txt'
    if not os.path.exists(corpus_path):
        # Fallback words
        return ["ꨆꨴꨯꩃ", "ꨟꨧꨮꩌ", "ꨓꨴꩀ", "ꨕꨠꩀ", "ꨀꨣꩌ", "ꨌꨰꩀ", "ꨙꩃ", "ꨚꨴꨯꨱꩃ", "ꨕꨫ"]
    
    with open(corpus_path, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Split by whitespace
    tokens = text.split()
    # Filter to retain mostly Cham words
    tokens = [t for t in tokens if len(t) >= 1]
    return tokens if tokens else ["ꨆꨴꨯꩃ", "ꨟꨧꨮꩌ"]

def main():
    PROJECT_ROOT = '.'
    fonts_dir = os.path.join(PROJECT_ROOT, 'data', 'fonts')
    output_dir = '/kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27'
    os.makedirs(output_dir, exist_ok=True)
    
    # Find fonts
    font_files = [os.path.join(fonts_dir, f) for f in os.listdir(fonts_dir) if f.endswith(('.ttf', '.otf'))]
    if not font_files:
        print("❌ No fonts found in data/fonts/")
        return
        
    validator = FontValidator(font_files)
    tokens = load_corpus_tokens()
    
    # Helper to generate random Cham word/sentence
    def gen_cham_word():
        return random.choice(tokens)
        
    def gen_cham_sentence(n_words=3):
        return ' '.join(gen_cham_word() for _ in range(n_words))
        
    # Stats trackers
    stats = {
        "latin_digit_count": 0,
        "bracket_count": 0,
        "period_count": 0,
        "comma_count": 0,
        "colon_count": 0,
        "semicolon_count": 0,
        "slash_count": 0,
        "dash_count": 0
    }
    
    def track_stats(text):
        for char in text:
            if char in LATIN_DIGITS:
                stats["latin_digit_count"] += 1
            elif char in "[]()":
                stats["bracket_count"] += 1
            elif char == '.':
                stats["period_count"] += 1
            elif char == ',':
                stats["comma_count"] += 1
            elif char == ':':
                stats["colon_count"] += 1
            elif char == ';':
                stats["semicolon_count"] += 1
            elif char == '/':
                stats["slash_count"] += 1
            elif char == '-':
                stats["dash_count"] += 1
                
    # 2. Dataset sampling functions
    def sample_normal():
        # 70% normal Cham lines
        n_words = random.randint(2, 6)
        return gen_cham_sentence(n_words)
        
    def sample_mixed():
        # 20% mixed Cham + digits / punctuation
        sentence = gen_cham_sentence(random.randint(2, 4))
        r = random.random()
        if r < 0.3:
            # append number
            sentence += f" {random.randint(0, 999)}"
        elif r < 0.6:
            # insert dash or slash
            sep = random.choice([" - ", " / ", " (2026) ", " [ref] "])
            sentence = gen_cham_sentence(random.randint(1, 2)) + sep + gen_cham_sentence(random.randint(1, 2))
        else:
            # punctuation at end or between
            sentence += random.choice([".", ",", " (ref).", " [ref];", ":"])
        return sentence
        
    def sample_hard():
        # 10% hard examples with brackets / periods / commas / line references
        r = random.random()
        if r < 0.4:
            # Choose from templates
            return random.choice(HARD_EXAMPLES_TEMPLATES)
        else:
            # Generate custom hard
            w1 = gen_cham_word()
            w2 = gen_cham_word()
            digit = random.randint(0, 99)
            hard_type = random.choice([
                f"{w1} ({digit})",
                f"{w1} [{digit}]",
                f"[{w1}] {w2}",
                f"{w1}, {w2}.",
                f"{w1} - {w2}",
                f"{w1}/{w2}",
                f"{digit}/{random.randint(1,12)}/{random.randint(2000, 2030)}"
            ])
            return hard_type
 
    # Generation loop
    num_samples = 8000
    n_normal = int(num_samples * 0.70) # 5600
    n_mixed = int(num_samples * 0.20)  # 1600
    n_hard = num_samples - n_normal - n_mixed # 800
    
    categories = (
        ['normal'] * n_normal +
        ['mixed'] * n_mixed +
        ['hard'] * n_hard
    )
    random.shuffle(categories)
    
    # Setup directories
    train_dir = os.path.join(output_dir, 'train')
    val_dir = os.path.join(output_dir, 'val')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)
    
    transform_pipeline = get_augmentation_pipeline()
    
    train_labels = []
    val_labels = []
    
    print("Generating train and val images...")
    # We split 8000 into 7200 train and 800 val
    for idx, cat in enumerate(categories):
        if cat == 'normal':
            text = sample_normal()
        elif cat == 'mixed':
            text = sample_mixed()
        else:
            text = sample_hard()
            
        track_stats(text)
        
        # Draw image
        selected_font = random.choice(font_files)
        clean_img = draw_text_strip(text, selected_font, img_width=320, img_height=48)
        
        # Apply augmentations (only train gets heavy augs, val gets light morph noise)
        is_val = (idx % 10 == 0)
        if not is_val:
            augmented_img = apply_augmentations(clean_img, transform_pipeline)
            save_subdir = train_dir
            label_prefix = 'train/'
        else:
            img_morph = apply_morphological_and_resolution_noise(clean_img)
            img_np = np.array(img_morph)
            augmented = transform_pipeline(image=img_np)
            final_img = Image.fromarray(augmented['image'])
            augmented_img = add_paper_noise(final_img, strength='light')
            save_subdir = val_dir
            label_prefix = 'val/'
            
        img_name = f"synth_{idx:06d}.png"
        save_path = os.path.join(save_subdir, img_name)
        augmented_img.save(save_path)
        
        visual_text = unicode_to_visual(text)
        label_line = f"{label_prefix}{img_name}\t{visual_text}\n"
        
        if not is_val:
            train_labels.append(label_line)
        else:
            val_labels.append(label_line)
            
    # Save label files
    with open(os.path.join(output_dir, 'train_label.txt'), 'w', encoding='utf-8') as f:
        f.writelines(train_labels)
    with open(os.path.join(output_dir, 'val_label.txt'), 'w', encoding='utf-8') as f:
        f.writelines(val_labels)
        
    # Write train label stats
    all_lengths = [len(unicode_to_visual(l.split('\t')[1].strip())) for l in train_labels]
    arr = np.array(all_lengths)
    
    bins_range = [(1, 10), (11, 25), (26, 50), (51, 80)]
    dist = {}
    for start_b, end_b in bins_range:
        dist[f"{start_b}-{end_b}"] = int(np.sum((arr >= start_b) & (arr <= end_b)))
        
    train_label_stats = {
        "count": len(all_lengths),
        "min": int(np.min(arr)) if len(arr) > 0 else 0,
        "max": int(np.max(arr)) if len(arr) > 0 else 0,
        "mean": round(float(np.mean(arr)), 2) if len(arr) > 0 else 0.0,
        "median": round(float(np.median(arr)), 2) if len(arr) > 0 else 0.0,
        "bins": dist
    }
    
    os.makedirs('output/v27_training', exist_ok=True)
    
    with open('/kaggle/working/paddleocr_cham_finetune/output/v27_training/train_label_stats.json', 'w', encoding='utf-8') as f:
        json.dump(train_label_stats, f, indent=2)
        
    with open('/kaggle/working/paddleocr_cham_finetune/output/v27_training/punctuation_coverage_stats.json', 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2)
        
    print("✅ Synthetic dataset generated successfully.")
    print("   Train size:", len(train_labels))
    print("   Val size:", len(val_labels))
    
    # 3. Generate locked evaluation splits
    eval_dir = os.path.join(output_dir, 'eval')
    os.makedirs(eval_dir, exist_ok=True)
    
    def generate_eval_split(split_name, size, sampler_fn, draw_fn=None):
        labels = []
        for i in range(size):
            text = sampler_fn()
            selected_font = random.choice(font_files)
            if draw_fn:
                img = draw_fn(text, selected_font)
            else:
                img = draw_text_strip(text, selected_font, img_width=320, img_height=48)
            img_name = f"{split_name}_{i:04d}.png"
            img.save(os.path.join(eval_dir, img_name))
            visual_text = unicode_to_visual(text)
            labels.append(f"eval/{img_name}\t{visual_text}\n")
            
        with open(os.path.join(output_dir, f"{split_name}_label.txt"), 'w', encoding='utf-8') as f:
            f.writelines(labels)
        print(f"   Generated evaluation split {split_name}: {size} images.")

    # samplers for eval splits
    def sampler_clean():
        return gen_cham_sentence(random.randint(2, 4))
        
    def sampler_hard_diacritic():
        sentence = gen_cham_sentence(random.randint(1, 2)) + " " + random.choice(["ꨨꨰꨳ", "ꨟꨧꨮꩌ", "ꨓꨌꨯꨱꩍ", "ꨝꨪꨗꩆ"]) + " " + gen_cham_sentence(random.randint(1, 2))
        return sentence
        
    def sampler_long():
        return gen_cham_sentence(random.randint(7, 10))
        
    def sampler_punctuation_digit():
        digit1 = random.randint(0, 999)
        digit2 = random.randint(2000, 2030)
        puncs = random.choice([
            f"({digit1})",
            f"[{digit1}]",
            f"{digit1}/{digit2}",
            f"{digit1} - {digit2}",
            ".,;:/-"
        ])
        return puncs
        
    def sampler_mixed_cham_punc():
        return sample_mixed()

    generate_eval_split('clean_cham_200', 200, sampler_clean)
    generate_eval_split('hard_diacritic_200', 200, sampler_hard_diacritic)
    generate_eval_split('long_line_200', 200, sampler_long)
    generate_eval_split('punctuation_digit_300', 300, sampler_punctuation_digit)
    generate_eval_split('mixed_cham_punctuation_300', 300, sampler_mixed_cham_punc)

if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/paddleocr_cham_finetune/scripts/evaluate_v27.py
import os
import sys
import json
import cv2
import paddle
import numpy as np

# Config Python paths
sys.path.insert(0, '/kaggle/working/PaddleOCR')
sys.path.insert(0, '/kaggle/working/paddleocr_cham_finetune')

from tools.infer.predict_rec import TextRecognizer
from scripts.generate_data import unicode_to_visual

def get_recognizer_args(model_dir, dict_path):
    import tools.infer.utility as utility
    sys_argv_backup = sys.argv
    sys.argv = [sys.argv[0]]
    args = utility.parse_args()
    sys.argv = sys_argv_backup
    
    args.rec_model_dir = model_dir
    args.rec_char_dict_path = dict_path
    args.rec_image_shape = '3, 48, 320'
    args.use_space_char = True
    args.use_gpu = True
    args.gpu_mem = 500
    return args

def calculate_cer(pred, gt):
    if not gt:
        return 1.0 if pred else 0.0
    # Levenshtein distance
    m, n = len(pred), len(gt)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
        
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred[i-1] == gt[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i-1][j] + 1, dp[i][j-1] + 1, dp[i-1][j-1] + 1)
    return dp[m][n] / len(gt)

def evaluate_splits():
    # Model directories
    v24_model_dir = '/kaggle/working/paddleocr_cham_finetune/output/rec_cham_inference_v24'
    v24_dict_path = '/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/v24_checkpoint/cham_dict.txt'
    
    v27_model_dir = '/kaggle/working/paddleocr_cham_finetune/output/rec_cham_inference_v27'
    v27_dict_path = '/kaggle/working/paddleocr_cham_finetune/data/cham_dict_v27.txt'
    
    # Initialize predictors
    print("Loading V23 Baseline Model...")
    v24_args = get_recognizer_args(v24_model_dir, v24_dict_path)
    v24_recognizer = TextRecognizer(v24_args)
    
    print("Loading V24 Candidate Model...")
    v27_args = get_recognizer_args(v27_model_dir, v27_dict_path)
    v27_recognizer = TextRecognizer(v27_args)
    
    splits = [
        'clean_cham_200',
        'hard_diacritic_200',
        'long_line_200',
        'punctuation_digit_300',
        'mixed_cham_punctuation_300'
    ]
    
    predictions = []
    summary = {}
    
    overall_cer_v24 = 0.0
    overall_cer_v27 = 0.0
    overall_count = 0
    
    cham_only_cer_v24 = 0.0
    cham_only_cer_v27 = 0.0
    cham_only_count = 0
    
    latin_digit_correct = 0
    latin_digit_total = 0
    punc_correct = 0
    punc_total = 0
    bracket_correct = 0
    bracket_total = 0
    
    # Required characters for verification
    required_latin_digits = set("0123456789")
    required_punctuation = set("[]().,;:/-")
    brackets = set("[]()")
    
    eval_root = '/kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27'
    
    for split in splits:
        split_label_path = os.path.join(eval_root, f"{split}_label.txt")
        if not os.path.exists(split_label_path):
            print(f"Warning: split label path {split_label_path} does not exist.")
            continue
            
        with open(split_label_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        split_cer_v24 = 0.0
        split_cer_v27 = 0.0
        split_acc_v24 = 0
        split_acc_v27 = 0
        split_count = len(lines)
        
        for line in lines:
            img_rel_path, gt = line.strip().split('\t')
            img_path = os.path.join(eval_root, img_rel_path)
            img = cv2.imread(img_path)
            if img is None:
                continue
                
            # Run inference
            res_v24_raw = v24_recognizer([img])[0][0]
            res_v27_raw = v27_recognizer([img])[0][0]
            
            res_v24 = res_v24_raw[0] if isinstance(res_v24_raw, (list, tuple)) else res_v24_raw
            res_v27 = res_v27_raw[0] if isinstance(res_v27_raw, (list, tuple)) else res_v27_raw
            
            # calculate CER
            cer_v24 = calculate_cer(res_v24, gt)
            cer_v27 = calculate_cer(res_v27, gt)
            
            split_cer_v24 += cer_v24
            split_cer_v27 += cer_v27
            
            if res_v24 == gt:
                split_acc_v24 += 1
            if res_v27 == gt:
                split_acc_v27 += 1
                
            # Track counts
            overall_cer_v24 += cer_v24
            overall_cer_v27 += cer_v27
            overall_count += 1
            
            # Cham-only evaluation
            has_latin_or_punc = any(c in required_latin_digits or c in required_punctuation for c in gt)
            if not has_latin_or_punc:
                cham_only_cer_v24 += cer_v24
                cham_only_cer_v27 += cer_v27
                cham_only_count += 1
                
            # Metric check for latin digits, punctuation, brackets
            for char in gt:
                if char in required_latin_digits:
                    latin_digit_total += 1
                    if char in res_v27:
                        latin_digit_correct += 1
                if char in required_punctuation:
                    punc_total += 1
                    if char in res_v27:
                        punc_correct += 1
                if char in brackets:
                    bracket_total += 1
                    if char in res_v27:
                        bracket_correct += 1
                        
            predictions.append({
                "split": split,
                "image_path": img_rel_path,
                "gt": gt,
                "pred_v24": [res_v24, float(res_v24_raw[1])] if isinstance(res_v24_raw, (list, tuple)) else [res_v24, 1.0],
                "pred_v27": [res_v27, float(res_v27_raw[1])] if isinstance(res_v27_raw, (list, tuple)) else [res_v27, 1.0],
                "cer_v24": round(cer_v24, 4),
                "cer_v27": round(cer_v27, 4),
                "important_cham_sign_changes": [],
                "latin_digit_correct": all(c in res_v27 for c in gt if c in required_latin_digits),
                "punctuation_correct": all(c in res_v27 for c in gt if c in required_punctuation)
            })
            
        summary[split] = {
            "cer_v24": round(split_cer_v24 / split_count, 4) if split_count > 0 else 0,
            "cer_v27": round(split_cer_v27 / split_count, 4) if split_count > 0 else 0,
            "acc_v24": round(split_acc_v24 / split_count, 4) if split_count > 0 else 0,
            "acc_v27": round(split_acc_v27 / split_count, 4) if split_count > 0 else 0
        }
        print(f"Split {split}: V23 CER={summary[split]['cer_v24']:.4f}, V24 CER={summary[split]['cer_v27']:.4f}")

    # Calculate overall metrics
    final_summary = {
        "splits": summary,
        "overall": {
            "cer_v24": round(overall_cer_v24 / overall_count, 4) if overall_count > 0 else 0,
            "cer_v27": round(overall_cer_v27 / overall_count, 4) if overall_count > 0 else 0,
            "cham_only_cer_v24": round(cham_only_cer_v24 / cham_only_count, 4) if cham_only_count > 0 else 0,
            "cham_only_cer_v27": round(cham_only_cer_v27 / cham_only_count, 4) if cham_only_count > 0 else 0,
            "latin_digit_accuracy": round(latin_digit_correct / latin_digit_total, 4) if latin_digit_total > 0 else 1.0,
            "punctuation_accuracy": round(punc_correct / punc_total, 4) if punc_total > 0 else 1.0,
            "bracket_accuracy": round(bracket_correct / bracket_total, 4) if bracket_total > 0 else 1.0,
            "cham_regression": round((cham_only_cer_v27 - cham_only_cer_v24) / cham_only_count, 4) if cham_only_count > 0 else 0.0
        }
    }
    
    # Save predictions and summary
    predictions_path = '/kaggle/working/paddleocr_cham_finetune/output/v27_training/predictions_v27.jsonl'
    summary_path = '/kaggle/working/paddleocr_cham_finetune/output/v27_training/eval_summary.json'
    
    with open(predictions_path, 'w', encoding='utf-8') as f:
        for p in predictions:
            f.write(json.dumps(p, ensure_ascii=False) + '\n')
            
    with open(summary_path, 'w', encoding='utf-8') as f:
        json.dump(final_summary, f, indent=2)
        
    print("✅ Evaluation complete.")
    print(f"Overall V23 CER: {final_summary['overall']['cer_v24']:.4f}")
    print(f"Overall V24 CER: {final_summary['overall']['cer_v27']:.4f}")

    # Generate comparison markdown
    comp_path = '/kaggle/working/paddleocr_cham_finetune/output/v27_training/v24_vs_v27_comparison.md'
    with open(comp_path, 'w', encoding='utf-8') as f:
        f.write("# V23 vs V24 Performance Comparison\n\n")
        f.write("## Overall Metrics\n\n")
        f.write("| Metric | V23 Baseline | V24 Candidate | Delta |\n")
        f.write("| --- | --- | --- | --- |\n")
        f.write(f"| Overall CER | {final_summary['overall']['cer_v24']:.2%} | {final_summary['overall']['cer_v27']:.2%} | {final_summary['overall']['cer_v27'] - final_summary['overall']['cer_v24']:.2%} |\n")
        f.write(f"| Cham-only CER | {final_summary['overall']['cham_only_cer_v24']:.2%} | {final_summary['overall']['cham_only_cer_v27']:.2%} | {final_summary['overall']['cham_only_cer_v27'] - final_summary['overall']['cham_only_cer_v24']:.2%} |\n")
        f.write(f"| Latin Digit Acc | - | {final_summary['overall']['latin_digit_accuracy']:.2%} | - |\n")
        f.write(f"| Punctuation Acc | - | {final_summary['overall']['punctuation_accuracy']:.2%} | - |\n")
        f.write(f"| Bracket Acc | - | {final_summary['overall']['bracket_accuracy']:.2%} | - |\n\n")
        
        f.write("## Split-wise Detail\n\n")
        f.write("| Split | V23 CER | V24 CER | V23 Accuracy | V24 Accuracy |\n")
        f.write("| --- | --- | --- | --- | --- |\n")
        for split, stats in summary.items():
            f.write(f"| {split} | {stats['cer_v24']:.2%} | {stats['cer_v27']:.2%} | {stats['acc_v24']:.2%} | {stats['acc_v27']:.2%} |\n")
            
    print("✅ Comparison markdown generated at:", comp_path)

if __name__ == '__main__':
    evaluate_splits()


## Phase 4: Run Data Generation, Dict Extension & Weight Surgery


In [ ]:
# Run scripts inside the project directory to ensure Cwd and paths are correct
%cd /kaggle/working/paddleocr_cham_finetune
!python3 scripts/dict_extension_v27.py
!python3 scripts/weight_surgery_v27.py
!python3 scripts/generate_data_v27.py
%cd /kaggle/working/


## Phase 5: Config & Training


In [ ]:
%%writefile /kaggle/working/paddleocr_cham_finetune/configs/rec_cham_v27.yml
Global:
  model_name: PP-OCRv4_mobile_rec
  debug: false
  use_gpu: true
  epoch_num: 100
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: /kaggle/working/paddleocr_cham_finetune/data/output/rec_cham_v27
  save_epoch_step: 10
  eval_batch_step:
  - 0
  - 500
  cal_metric_during_train: true
  pretrained_model: /kaggle/working/paddleocr_cham_finetune/data/output_v27_temp/rec_cham_best_model/iter_epoch_200
  checkpoints: null
  save_inference_dir: null
  use_visualdl: false
  infer_img: doc/imgs_words/ch/word_1.jpg
  character_dict_path: /kaggle/working/paddleocr_cham_finetune/data/cham_dict_v27.txt
  max_text_length: 80
  infer_mode: false
  use_space_char: true
  distributed: true
  save_res_path: /kaggle/working/paddleocr_cham_finetune/data/output/rec_cham_v27
  d2s_train_image_shape:
  - 3
  - 48
  - 320
Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0002
    warmup_epoch: 2
  regularizer:
    name: L2
    factor: 3.0e-05
Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform: null
  Backbone:
    name: PPLCNetV3
    scale: 0.95
  Head:
    name: MultiHead
    head_list:
    - CTCHead:
        Neck:
          name: svtr
          dims: 120
          depth: 2
          hidden_dims: 120
          kernel_size:
          - 1
          - 3
          use_guide: true
        Head:
          fc_decay: 1.0e-05
    - NRTRHead:
        nrtr_dim: 384
        max_text_length: 80
Loss:
  name: MultiLoss
  loss_config_list:
  - CTCLoss: null
  - NRTRLoss: null
PostProcess:
  name: CTCLabelDecode
Metric:
  name: RecMetric
  main_indicator: acc
Train:
  dataset:
    name: MultiScaleDataSet
    ds_width: false
    data_dir: /kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27
    ext_op_transform_idx: 1
    label_file_list:
    - /kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27/train_label.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - RecConAug:
        prob: 0.5
        ext_data_num: 2
        image_shape:
        - 48
        - 320
        - 3
        max_text_length: 80
    - RecAug: null
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  sampler:
    name: MultiScaleSampler
    scales:
    - - 320
      - 32
    - - 320
      - 48
    - - 320
      - 64
    first_bs: 24
    fix_bs: false
    divided_factor:
    - 8
    - 16
    is_training: true
  loader:
    shuffle: true
    batch_size_per_card: 24
    drop_last: true
    num_workers: 2
Eval:
  dataset:
    name: SimpleDataSet
    data_dir: /kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27
    label_file_list:
    - /kaggle/working/paddleocr_cham_finetune/data/cham_synthetic_v27/val_label.txt
    transforms:
    - DecodeImage:
        img_mode: BGR
        channel_first: false
    - MultiLabelEncode:
        gtc_encode: NRTRLabelEncode
    - RecResizeImg:
        image_shape:
        - 3
        - 48
        - 320
    - KeepKeys:
        keep_keys:
        - image
        - label_ctc
        - label_gtc
        - length
        - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 24
    num_workers: 2


In [ ]:
# Execute training with GPU T4x2 launch config
import paddle
gpu_count = paddle.device.cuda.device_count()
print("GPU Count:", gpu_count)

if gpu_count > 1:
    !cd /kaggle/working/PaddleOCR && python3 -m paddle.distributed.launch --gpus '0,1' tools/train.py -c /kaggle/working/paddleocr_cham_finetune/configs/rec_cham_v27.yml -o Global.use_gpu=True Global.pretrained_model=/kaggle/working/paddleocr_cham_finetune/data/output_v27_temp/rec_cham_best_model/iter_epoch_200
else:
    !cd /kaggle/working/PaddleOCR && python3 tools/train.py -c /kaggle/working/paddleocr_cham_finetune/configs/rec_cham_v27.yml -o Global.use_gpu=True Global.pretrained_model=/kaggle/working/paddleocr_cham_finetune/data/output_v27_temp/rec_cham_best_model/iter_epoch_200


## Phase 6: Export Inference Model


In [ ]:
# Export V23 baseline model to inference format
!cd /kaggle/working/PaddleOCR && python3 tools/export_model.py -c /kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/v24_checkpoint/config.yml -o Global.pretrained_model=/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/v24_checkpoint/iter_epoch_200 Global.character_dict_path=/kaggle/input/datasets/gustavnguyen/cham-ocr-v5-assets/v24_checkpoint/cham_dict.txt Global.save_inference_dir=/kaggle/working/paddleocr_cham_finetune/output/rec_cham_inference_v24

# Export V24 model to inference format
!cd /kaggle/working/PaddleOCR && python3 tools/export_model.py -c /kaggle/working/paddleocr_cham_finetune/configs/rec_cham_v27.yml -o Global.pretrained_model=/kaggle/working/paddleocr_cham_finetune/data/output/rec_cham_v27/best_accuracy Global.checkpoints=null Global.save_inference_dir=/kaggle/working/paddleocr_cham_finetune/output/rec_cham_inference_v27


## Phase 7: Run Evaluation & Comparison


In [ ]:
# Run the evaluation script inside the project directory
%cd /kaggle/working/paddleocr_cham_finetune
!python3 scripts/evaluate_v27.py
%cd /kaggle/working/


## Phase 8: Package Results & Generate Model Card


In [ ]:
import os, hashlib, json

# Read dict and checkpoint hash
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

dict_hash = sha256('/kaggle/working/paddleocr_cham_finetune/data/cham_dict_v27.txt')
model_hash = sha256('/kaggle/working/paddleocr_cham_finetune/output/rec_cham_inference_v27/inference.pdiparams')

# Load eval stats
with open('/kaggle/working/paddleocr_cham_finetune/output/v27_training/eval_summary.json', 'r') as f:
    eval_stats = json.load(f)

card_content = f"""# Model Card — rec_cham_inference_v27

## Model Overview
V24 OCR Recognizer patch model. Fine-tuned from V23 baseline to add Latin digits and common punctuation.

## Architecture
- Backbone: PPLCNetV3
- Neck: SVTR Neck
- Head: MultiHead (CTC + NRTR)
- Output size: 103 classes (CTC), 107 classes (GTC)

## Metadata
- Base Checkpoint: V23 best accuracy (iter_epoch_200)
- Dict Hash: {dict_hash}
- inference.pdiparams Hash: {model_hash}
- Training dataset size: 8000 images (70% normal Cham, 20% mixed, 10% hard examples)
- Epochs: 100 epochs

## Evaluation Summary
- Overall CER: {eval_stats['overall']['cer_v27']:.2%} (V23 Baseline: {eval_stats['overall']['cer_v24']:.2%})
- Cham-only CER: {eval_stats['overall']['cham_only_cer_v27']:.2%} (V23 Baseline: {eval_stats['overall']['cham_only_cer_v24']:.2%})
- Latin Digit Accuracy: {eval_stats['overall']['latin_digit_accuracy']:.2%}
- Punctuation Accuracy: {eval_stats['overall']['punctuation_accuracy']:.2%}
- Bracket Accuracy: {eval_stats['overall']['bracket_accuracy']:.2%}
- Regression vs V23: {eval_stats['overall']['cham_regression']:.2%}

## Added Characters
0 1 2 3 4 5 6 7 8 9 [ ] ( ) . , : ; - /
"""

with open('/kaggle/working/paddleocr_cham_finetune/output/v27_training/v27_model_card.md', 'w', encoding='utf-8') as f:
    f.write(card_content)
    
print("Model card generated successfully.")


In [ ]:
import shutil
pack_dir = '/kaggle/working/v27_training_evidence'
os.makedirs(pack_dir, exist_ok=True)

# Copy files
output_src = '/kaggle/working/paddleocr_cham_finetune/output/v27_training'
shutil.copy2(os.path.join(output_src, 'dict_extension_report.json'), os.path.join(pack_dir, 'dict_extension_report.json'))
shutil.copy2(os.path.join(output_src, 'head_extension_report.json'), os.path.join(pack_dir, 'head_extension_report.json'))
shutil.copy2(os.path.join(output_src, 'train_label_stats.json'), os.path.join(pack_dir, 'train_label_stats.json'))
shutil.copy2(os.path.join(output_src, 'punctuation_coverage_stats.json'), os.path.join(pack_dir, 'punctuation_coverage_stats.json'))
shutil.copy2(os.path.join(output_src, 'eval_summary.json'), os.path.join(pack_dir, 'eval_summary.json'))
shutil.copy2(os.path.join(output_src, 'v24_vs_v27_comparison.md'), os.path.join(pack_dir, 'v24_vs_v27_comparison.md'))
shutil.copy2(os.path.join(output_src, 'predictions_v27.jsonl'), os.path.join(pack_dir, 'predictions_v27.jsonl'))
shutil.copy2(os.path.join(output_src, 'v27_model_card.md'), os.path.join(pack_dir, 'v27_model_card.md'))
shutil.copy2('/kaggle/working/paddleocr_cham_finetune/configs/rec_cham_v27.yml', os.path.join(pack_dir, 'rec_cham_v27.yml'))

# Copy train logs
shutil.copy2('/kaggle/working/paddleocr_cham_finetune/data/output/rec_cham_v27/train.log', os.path.join(pack_dir, 'train_log.txt'))

# Make zip archive
shutil.make_archive('/kaggle/working/v27_training_evidence', 'zip', pack_dir)
# Copy zip to expected output location
os.makedirs('/kaggle/working/paddleocr_cham_finetune/output/v27_training', exist_ok=True)
shutil.copy2('/kaggle/working/v27_training_evidence.zip', '/kaggle/working/paddleocr_cham_finetune/output/v27_training/v27_training_evidence.zip')
print("✅ Output packaged successfully.")

# Clean up large directories to reduce output size and keep only packaged zip and models
for path in ['/kaggle/working/PaddleOCR', '/kaggle/working/paddleocr_cham_finetune/data', '/kaggle/working/v27_training_evidence']:
    if os.path.exists(path):
        print(f"Cleaning up {path}...")
        try:
            shutil.rmtree(path)
        except Exception as e:
            print(f"Error cleaning {path}: {e}")
print("✅ Large directories cleaned up.")
